# Lesson 7: Images and the first CNN

Goes with the Lesson 7 page of the Forkcast course. Images as (N, C, H, W) tensors, convolution, pooling, a small CNN, and cross-entropy.

Runtime: pick a **GPU** (Runtime → Change runtime type). The synthetic shapes train on a CPU; the dish photos at the end want the GPU.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/vision.py`

The shape generator and loaders are provided. Fill in `conv_output_size`, `SmallCNN`, `accuracy` and `cross_entropy_by_hand` as the page walks through them, re-running the cell after each edit.

In [ ]:
%%writefile forkcast/vision.py
"""Lesson 7. Images as tensors, and the first convolutional network.

``make_shapes`` (provided) draws synthetic pictures of four shapes so the tests
run offline; the notebook trains the same network on real dish photos.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

SHAPE_CLASSES = ["circle", "square", "triangle", "cross"]


def make_shapes(n: int = 1000, seed: int = 0, size: int = 32) -> tuple[Tensor, Tensor]:
    """Return ``(images, labels)``: images (n, 3, size, size) float32 in [0, 1], labels (n,) int64.

    Each image is one coloured shape on a dark, lightly noisy background, at a random
    position and scale. Classes in SHAPE_CLASSES order: circle, square, triangle, cross.
    """
    g = torch.Generator().manual_seed(seed)
    labels = torch.randint(0, 4, (n,), generator=g)
    yy, xx = torch.meshgrid(torch.arange(size, dtype=torch.float32), torch.arange(size, dtype=torch.float32), indexing="ij")
    images = torch.zeros(n, 3, size, size)
    for i in range(n):
        cx, cy = (size * (0.3 + 0.4 * torch.rand(2, generator=g))).tolist()
        r = size * (0.12 + 0.13 * torch.rand(1, generator=g).item())
        dx, dy = xx - cx, yy - cy
        k = labels[i].item()
        if k == 0:
            mask = dx ** 2 + dy ** 2 <= r ** 2
        elif k == 1:
            mask = (dx.abs() <= r) & (dy.abs() <= r)
        elif k == 2:
            mask = (dy >= -r) & (dy <= r) & (dx.abs() <= (r - dy) * 0.6)
        else:
            mask = ((dx.abs() <= r * 0.3) & (dy.abs() <= r)) | ((dy.abs() <= r * 0.3) & (dx.abs() <= r))
        colour = 0.5 + 0.5 * torch.rand(3, generator=g)
        images[i] = 0.08 * torch.rand(3, size, size, generator=g)
        images[i][:, mask] = colour[:, None]
    return images.clamp(0, 1), labels


def shape_loaders(n_train: int = 1000, n_val: int = 400, batch_size: int = 64, size: int = 32) -> tuple[DataLoader, DataLoader]:
    """Loaders over TensorDatasets of synthetic shapes (train seed 0, validation seed 1)."""
    x_tr, y_tr = make_shapes(n_train, seed=0, size=size)
    x_val, y_val = make_shapes(n_val, seed=1, size=size)
    return (DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=True),
            DataLoader(TensorDataset(x_val, y_val), batch_size=batch_size, shuffle=False))


def conv_output_size(size: int, kernel: int, stride: int = 1, padding: int = 0) -> int:
    """Spatial size after a convolution or pooling layer: floor((size + 2p - k) / s) + 1."""
    raise NotImplementedError("Lesson 7, step 3: the formula in the docstring, with integer division")


def accuracy(logits: Tensor, y: Tensor) -> float:
    """Fraction of rows whose highest logit is the true class. (N, C), (N,) -> float."""
    raise NotImplementedError("Lesson 7, step 6: (logits.argmax(dim=1) == y).float().mean().item()")


def cross_entropy_by_hand(logits: Tensor, y: Tensor) -> Tensor:
    """What nn.CrossEntropyLoss computes: minus the log-probability of the true class, averaged.

    Use F.log_softmax over the class dimension, pick each row's true-class entry, negate, mean.
    """
    raise NotImplementedError("Lesson 7, step 6: logp = F.log_softmax(logits, dim=1); return -logp[torch.arange(len(y)), y].mean()")


class SmallCNN(nn.Module):
    """Two conv blocks, then a small classifier head.

    Block: Conv2d(3x3, padding=1) -> BatchNorm2d -> ReLU -> MaxPool2d(2). Channels 3 -> 16 -> 32.
    After two pools a size x size image is (size/4) x (size/4), so the head's first
    Linear takes 32 * (size // 4) ** 2 inputs. The head ends in Linear(64, n_classes), no activation:
    those are the logits.
    """

    def __init__(self, n_classes: int = 4, size: int = 32):
        super().__init__()
        raise NotImplementedError("Lesson 7, step 5: self.features = nn.Sequential(...); self.head = nn.Sequential(nn.Flatten(), nn.Linear(...), nn.ReLU(), nn.Linear(64, n_classes))")

    def forward(self, x: Tensor) -> Tensor:
        """(N, 3, size, size) -> (N, n_classes)"""
        raise NotImplementedError("Lesson 7, step 5: return self.head(self.features(x))")


@torch.no_grad()
def accuracy_loader(model: nn.Module, loader, device: str | torch.device = "cpu") -> float:
    """Accuracy over every batch in ``loader``."""
    model.eval()
    correct, seen = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(dim=1) == y).sum().item()
        seen += len(y)
    return correct / seen

## 3. Look at the pictures

In [ ]:
import importlib, torch
from torch import nn
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from forkcast import vision, train
importlib.reload(vision)
from forkcast.model import count_parameters

images, labels = vision.make_shapes(16, seed=0)
print(images.shape, images.dtype, labels)
print("per-channel mean:", images.mean(dim=(0, 2, 3)))
plt.figure(figsize=(10, 3)); plt.axis("off"); plt.imshow(make_grid(images, nrow=8).permute(1, 2, 0))
print([vision.SHAPE_CLASSES[k] for k in labels])

## 4. Shape arithmetic, checked against a real layer

In [ ]:
importlib.reload(vision)
for size, k, s, p in [(32, 3, 1, 1), (32, 3, 2, 1), (32, 5, 1, 0), (64, 7, 2, 3)]:
    real = nn.Conv2d(1, 1, k, stride=s, padding=p)(torch.zeros(1, 1, size, size)).shape[-1]
    print(size, k, s, p, "->", vision.conv_output_size(size, k, s, p), real)

### Slide a kernel by hand

The vertical-edge kernel from the page, applied to the red channel of a square.

In [ ]:
edge = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    edge.weight[:] = torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]])
square = images[labels == 1][:1, :1]              # one image, red channel only: (1, 1, 32, 32)
response = edge(square)
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(square[0, 0], cmap="gray"); ax[0].set_title("input"); ax[1].imshow(response[0, 0].detach(), cmap="coolwarm"); ax[1].set_title("vertical edges")
print(square.shape, "->", response.shape)

## 5. The network, shape by shape

In [ ]:
importlib.reload(vision)
model = vision.SmallCNN()
print(model.features(images).shape)          # (16, 32, 8, 8)
print(model(images).shape)                   # (16, 4)
print(count_parameters(model))               # 136580

## 6. `accuracy` and `cross_entropy_by_hand`

In [ ]:
importlib.reload(vision)
logits = torch.tensor([[2.0, 0.1, 0.0, 0.0], [0.0, 3.0, 0.0, 0.0], [0.0, 0.0, 0.5, 0.4], [1.0, 0.0, 0.0, 0.0]])
y = torch.tensor([0, 1, 2, 3])
print(vision.accuracy(logits, y))
print(vision.cross_entropy_by_hand(logits, y), nn.CrossEntropyLoss()(logits, y))

## 7. Train on the shapes

In [ ]:
train_loader, val_loader = vision.shape_loaders(n_train=1000, n_val=400)
torch.manual_seed(0)
model = vision.SmallCNN()
history = train.fit_loader(model, train_loader, val_loader, epochs=6, lr=1e-3, loss_fn=nn.CrossEntropyLoss())
print([round(v, 2) for v in history["val"]])
print("validation accuracy:", vision.accuracy_loader(model, val_loader))
out = model(images[:1]); print(out, torch.softmax(out, dim=1), "true:", vision.SHAPE_CLASSES[labels[0]])

### Break it on purpose, twice

In [ ]:
try:
    model(images.permute(0, 2, 3, 1))                # channels last
except RuntimeError as err:
    print("RuntimeError:", str(err)[:140])

torch.manual_seed(0)
wrong = vision.SmallCNN()
train.fit_loader(wrong, train_loader, val_loader, epochs=6, lr=1e-3, loss_fn=lambda logits, yy: nn.MSELoss()(logits[:, 0], yy.float()))
print("MSE on the class index: accuracy", vision.accuracy_loader(wrong, val_loader))

## 8. Real dish photos

Ten classes of Food-101, streamed from the Hugging Face Hub so nothing large is downloaded, resized to 64×64. A few minutes on the GPU.

In [ ]:
!pip install -q datasets
from datasets import load_dataset
from torchvision.transforms import v2
from torch.utils.data import DataLoader, TensorDataset

KEEP = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]                 # ten of the 101 classes
stream = load_dataset("ethz/food101", split="train", streaming=True)
names = stream.features["label"].names
to_tensor = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Resize((64, 64), antialias=True)])
xs, ys = [], []
for ex in stream:
    if ex["label"] in KEEP:
        xs.append(to_tensor(ex["image"].convert("RGB"))); ys.append(KEEP.index(ex["label"]))
    if len(xs) >= 2000:
        break
X = torch.stack(xs); Y = torch.tensor(ys)
print(X.shape, [names[k] for k in KEEP])
mean, std = X.mean(dim=(0, 2, 3)), X.std(dim=(0, 2, 3))
Xn = (X - mean[None, :, None, None]) / std[None, :, None, None]
food_train = DataLoader(TensorDataset(Xn[:1600], Y[:1600]), batch_size=64, shuffle=True, drop_last=True)
food_val = DataLoader(TensorDataset(Xn[1600:], Y[1600:]), batch_size=64)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
food_model = vision.SmallCNN(n_classes=10, size=64)
h = train.fit_loader(food_model, food_train, food_val, epochs=15, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device, patience=5)
print("validation accuracy on real dishes:", round(vision.accuracy_loader(food_model, food_val, device), 3), "(chance is 0.10)")
plt.plot(h["train"], label="train"); plt.plot(h["val"], label="validation"); plt.legend();

## 9. Definition of done

Expected: `7 passed`, then `55 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_07.py -q

In [ ]:
!pytest -m fast -q

## 10. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 7: images and the first CNN"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main